# AR・MA・ARIMA・SARIMA・ARIMAX（最小構成版）

シンプルな合成データ1つで、時系列モデルの系譜を体験します。  
各セクションは5〜10行のコードで完結します。


## 0. 準備

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams['axes.unicode_minus'] = False


## 1. データ生成

「トレンド＋季節性＋自己相関ノイズ」を持つ5年分の月次データを作る。  
$y_t = 50 + 0.3t + 10\sin(2\pi t/12) + e_t$, $e_t = 0.6 e_{t-1} + \varepsilon_t$


In [ ]:
np.random.seed(0)
n = 60
t = np.arange(n)
trend  = 0.3 * t
season = 10 * np.sin(2 * np.pi * t / 12)

e = np.zeros(n)
noise = np.random.normal(0, 1.5, n)
for i in range(1, n):
    e[i] = 0.6 * e[i-1] + noise[i]   # AR(1)的な自己相関ノイズ

y = 50 + trend + season + e
dates = pd.date_range('2019-01-01', periods=n, freq='MS')
s = pd.Series(y, index=dates, name='y')
s.index.freq = 'MS'

train, test = s[:-12], s[-12:]   # 直近12ヶ月をテストに
s.plot(figsize=(8, 3), title='合成時系列データ')
plt.tight_layout(); plt.show()


## 2. 定常性とACF/PACF

ADF検定で非定常性を確認し、ACF/PACFでモデルの手がかりを得る。


In [ ]:
adf_p = adfuller(s)[1]
print(f'ADF検定 p値 = {adf_p:.4f}  → {"非定常" if adf_p > 0.05 else "定常"}')

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
plot_acf(s, lags=24, ax=axes[0])
plot_pacf(s, lags=24, ax=axes[1])
plt.tight_layout(); plt.show()


**観察：** ACF が緩やかに減衰し、lag=12付近で再び持ち上がる（季節性の手がかり）。

## 3. AR(p) ― 自己回帰

$y_t = c + \phi_1 y_{t-1} + \cdots + \phi_p y_{t-p} + \varepsilon_t$


In [ ]:
m_ar = ARIMA(train, order=(1, 0, 0)).fit()
print(m_ar.summary().tables[1])
print(f'AIC = {m_ar.aic:.1f}')


## 4. MA(q) ― 移動平均

$y_t = \mu + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \cdots$  
AR は「過去の値」、MA は「過去の予測誤差（ショック）」に依存する点が違う。


In [ ]:
m_ma = ARIMA(train, order=(0, 0, 1)).fit()
print(m_ma.summary().tables[1])
print(f'AIC = {m_ma.aic:.1f}')


## 5. ARIMA(p,d,q) ― 差分を組み込む

$d$ は非定常性に対処する差分の階数。AIC で $(p,q)$ を選ぶ。


In [ ]:
best_aic, best_order = np.inf, None
for p in range(3):
    for q in range(3):
        aic = ARIMA(train, order=(p, 1, q)).fit().aic
        if aic < best_aic:
            best_aic, best_order = aic, (p, 1, q)

m_arima = ARIMA(train, order=best_order).fit()
print(f'最良: ARIMA{best_order}  AIC={best_aic:.1f}')


In [ ]:
pred_arima = m_arima.get_forecast(12).predicted_mean

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(train.index[-24:], train[-24:], label='訓練データ')
ax.plot(test.index, test, label='実測値', color='#27ae60')
ax.plot(test.index, pred_arima, '--', label=f'ARIMA{best_order} 予測', color='#e74c3c')
ax.legend(); ax.set_title('ARIMA の予測')
plt.tight_layout(); plt.show()


**観察：** トレンドは捉えられるが、季節の波（夏冬の上下）は再現できていない。

## 6. SARIMA(p,d,q)(P,D,Q)s ― 季節性を追加

季節周期 $s=12$ を明示的に組み込み、季節差分・季節AR/MAを追加する。


In [ ]:
m_sarima = SARIMAX(train, order=best_order,
                   seasonal_order=(1, 1, 1, 12)).fit(disp=False)
print(f'SARIMA AIC = {m_sarima.aic:.1f}　（ARIMA AIC = {best_aic:.1f}）')


In [ ]:
pred_sarima = m_sarima.get_forecast(12).predicted_mean

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(test.index, test, label='実測値', color='#27ae60', marker='o')
ax.plot(test.index, pred_arima,  '--', label='ARIMA',  color='#95a5a6')
ax.plot(test.index, pred_sarima, '--', label='SARIMA', color='#e74c3c')
ax.legend(); ax.set_title('ARIMA vs SARIMA：季節性の捕捉')
plt.tight_layout(); plt.show()


**観察：** SARIMAは季節の波を再現できている。AICもARIMAより小さい（改善）。

## 7. ARIMAX ― 外生変数を追加する

ある時点から構造が変わった、という外生変数（ダミー）を追加する例。  
$y_t$ の式に $\beta \cdot \text{dummy}_t$ を加える。


In [ ]:
dummy = pd.Series(np.where(t < 30, 0, 1), index=dates, name='dummy')

m_arimax = SARIMAX(train, exog=dummy[:-12], order=best_order,
                   seasonal_order=(1, 1, 1, 12)).fit(disp=False)
print(m_arimax.summary().tables[1].tail(3))
print(f'ARIMAX AIC = {m_arimax.aic:.1f}　（SARIMA AIC = {m_sarima.aic:.1f}）')


---
## まとめ

| モデル | 追加した要素 | 効果 |
|--------|------------|------|
| AR(p) | 過去の値 | 短期の自己相関を捉える |
| MA(q) | 過去のショック | 一時的な変動の影響を捉える |
| ARIMA | 差分 $d$ | トレンド（非定常性）に対処 |
| SARIMA | 季節成分 $(P,D,Q)_s$ | 周期的な変動を捉える |
| ARIMAX | 外生変数 | 構造変化やイベントの効果を組み込む |

### 演習
1. `order=(1,1,1)` の季節差分を `seasonal_order=(1,1,1,12)` から `(0,1,1,12)` に変えると AIC はどう変わるか？
2. ダミー変数の切り替え時点（`t < 30`）を `t < 40` に変えると ARIMAX の係数はどう変わるか？
